# Chapter 35: Monocular SLAM

<a href="../lite/lab/index.html?path=ch35_monocular_slam.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.linalg import svd, null_space

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

One eye is not enough for depth. Cover one eye and try to catch a ball. You will
probably miss. A monocular SLAM system faces the same problem every frame. It can
recover relative structure (things twice as far look half as big), but not absolute
scale. A room could be 3 meters or 30 meters across, and the single camera cannot tell.

In this chapter we will build the mathematics and code to understand:
- Why a single image **cannot** determine depth
- How **scale ambiguity** is the fundamental limitation
- How **two view initialization** recovers structure up to scale
- How **triangulation** accuracy depends on geometry
- The common **failure modes** of monocular SLAM

```{admonition} What you will build
:class: tip

- Prove that a single camera cannot determine absolute scale
- Initialize a monocular SLAM map from two views using the essential matrix
- Measure how triangulation accuracy depends on the baseline to depth ratio
- Diagnose failure modes: pure rotation, forward motion, scale drift

**Real world application:** Monocular SLAM powers smartphone AR (ARKit, ARCore). After this chapter, you will understand why your phone needs to move sideways before it can place virtual objects accurately.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **ORB-SLAM3 (monocular mode)** | The most widely used monocular SLAM system |
| **DSO (Direct Sparse Odometry)** | Direct (non-feature) monocular odometry |
| **OpenVSLAM / stella_vslam** | Open source visual SLAM supporting monocular, stereo, RGBD |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

## 35.1 Ill Posed Nature

Depth from a **single image** is fundamentally **ill posed**. A 3D point $\mathbf{X}$
projects to a pixel $\mathbf{u}$ via

$$\mathbf{u} = \pi(K \mathbf{X}) = K \begin{bmatrix} X/Z \\ Y/Z \\ 1 \end{bmatrix}$$

Any point along the ray from the camera center through pixel $\mathbf{u}$ produces the
same observation. This means infinitely many 3D scenes can produce the exact same image.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(42)
n_points = 8                     # number of 3D points
focal_length = 400.0             # pixels
cx, cy = 320.0, 240.0            # principal point
depth_configs = [5.0, 10.0, 20.0] # three different depth scalings
# ─────────────────────────────────────────────────────────────────────────────

K = np.array([[focal_length, 0, cx],
              [0, focal_length, cy],
              [0, 0, 1.0]])

# Generate 3D points at unit depth, then scale to different depths
pts_unit = np.random.uniform(-1, 1, (n_points, 2))  # x, y in [-1,1]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pixel_coords_all = []

for i, d in enumerate(depth_configs):
    # Create 3D points at depth d
    pts_3d = np.column_stack([pts_unit * d, np.full(n_points, d)])
    # Project: u = K * [X/Z, Y/Z, 1]
    pts_norm = pts_3d[:, :2] / pts_3d[:, 2:3]
    pixels = pts_norm @ K[:2, :2].T + K[:2, 2]
    pixel_coords_all.append(pixels)
    
    ax = axes[i]
    ax.scatter(pixels[:, 0], pixels[:, 1], c='steelblue', s=60, zorder=5)
    ax.set_xlim(0, 640); ax.set_ylim(480, 0)
    ax.set_title(f'Depth = {d:.0f} m', fontsize=12)
    ax.set_xlabel('u (px)'); ax.set_ylabel('v (px)')
    ax.set_aspect('equal')

plt.suptitle('Three different 3D scenes produce the SAME image', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Verify pixel coordinates are identical
for i in range(1, len(depth_configs)):
    diff = np.max(np.abs(pixel_coords_all[0] - pixel_coords_all[i]))
    print(f'Max pixel difference between depth={depth_configs[0]} and depth={depth_configs[i]}: {diff:.2e}')

In [ ]:
# Visualize the ray ambiguity in 3D
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

colors = ['steelblue', 'tomato', 'forestgreen']
for i, d in enumerate(depth_configs):
    pts_3d = np.column_stack([pts_unit * d, np.full(n_points, d)])
    ax.scatter(pts_3d[:, 0], pts_3d[:, 1], pts_3d[:, 2],
               c=colors[i], s=50, label=f'Scene at depth {d:.0f}', depthshade=True)

# Draw rays from camera center to points at max depth
for j in range(n_points):
    d_max = max(depth_configs)
    end = np.array([pts_unit[j, 0] * d_max, pts_unit[j, 1] * d_max, d_max])
    ax.plot([0, end[0]], [0, end[1]], [0, end[2]], 'gray', alpha=0.3, linewidth=0.8)

ax.scatter([0], [0], [0], c='black', s=100, marker='^', label='Camera center')
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z (depth)')
ax.set_title('All points along each ray project to the same pixel', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

**Key insight:** From a single camera, every pixel defines a **ray** in 3D. Any point
along that ray produces the same observation. Without additional information (a second
view, known object sizes, or a depth sensor), depth is completely unrecoverable.

## 35.2 Scale Ambiguity

Even with **two views**, monocular SLAM suffers from **scale ambiguity**. If we
scale all 3D points by a factor $s$ and simultaneously scale the camera baseline
by $s$, the projections in both images remain identical:

$$\pi(K [R | t] \mathbf{X}) = \pi(K [R | s\,t] \, s\,\mathbf{X})$$

This is because the projection divides by the $Z$ coordinate, canceling $s$.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(42)
scale_factor = 3.0               # scale the scene by this factor
n_scene_pts = 12                 # number of 3D points
baseline_x = 0.5                 # baseline between two cameras (meters)
# ─────────────────────────────────────────────────────────────────────────────

# Camera intrinsics
K = np.array([[400, 0, 320],
              [0, 400, 240],
              [0, 0, 1.0]])

# Generate 3D points: original scene
pts_3d_orig = np.random.uniform(-2, 2, (n_scene_pts, 3))
pts_3d_orig[:, 2] = np.abs(pts_3d_orig[:, 2]) + 3  # ensure positive depth

# Scaled scene
pts_3d_scaled = pts_3d_orig * scale_factor

# Camera 1: at origin. Camera 2: translated by baseline.
t_orig = np.array([baseline_x, 0, 0])
t_scaled = t_orig * scale_factor
R = np.eye(3)  # no rotation for simplicity

def project(K, R, t, pts_3d):
    """Project 3D points to 2D using K[R|t]."""
    pts_cam = (R @ pts_3d.T).T + t  # transform to camera frame
    pts_norm = pts_cam[:, :2] / pts_cam[:, 2:3]
    pixels = pts_norm @ K[:2, :2].T + K[:2, 2]
    return pixels

# Project original scene
px1_orig = project(K, R, np.zeros(3), pts_3d_orig)
px2_orig = project(K, R, t_orig, pts_3d_orig)

# Project scaled scene
px1_scaled = project(K, R, np.zeros(3), pts_3d_scaled)
px2_scaled = project(K, R, t_scaled, pts_3d_scaled)

# Compare
print('=== SCALE AMBIGUITY DEMONSTRATION ===')
print(f'Original scene: {n_scene_pts} points, baseline = {baseline_x:.2f} m')
print(f'Scaled scene:   {n_scene_pts} points, baseline = {baseline_x * scale_factor:.2f} m')
print(f'Scale factor: {scale_factor}')
print(f'\nMax pixel difference in Camera 1: {np.max(np.abs(px1_orig - px1_scaled)):.2e}')
print(f'Max pixel difference in Camera 2: {np.max(np.abs(px2_orig - px2_scaled)):.2e}')
print('\nBoth scenes produce IDENTICAL images in BOTH cameras.')

In [ ]:
# Side by side: the two scenes look completely different in 3D
fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(pts_3d_orig[:, 0], pts_3d_orig[:, 1], pts_3d_orig[:, 2],
            c='steelblue', s=60)
ax1.scatter([0, t_orig[0]], [0, 0], [0, 0], c='tomato', s=100, marker='^')
ax1.plot([0, t_orig[0]], [0, 0], [0, 0], 'tomato', linewidth=2)
ax1.set_title(f'Original (baseline={baseline_x:.1f}m)', fontsize=12)
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(pts_3d_scaled[:, 0], pts_3d_scaled[:, 1], pts_3d_scaled[:, 2],
            c='forestgreen', s=60)
ax2.scatter([0, t_scaled[0]], [0, 0], [0, 0], c='tomato', s=100, marker='^')
ax2.plot([0, t_scaled[0]], [0, 0], [0, 0], 'tomato', linewidth=2)
ax2.set_title(f'Scaled x{scale_factor:.0f} (baseline={baseline_x*scale_factor:.1f}m)', fontsize=12)
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')

plt.suptitle('Two physically different scenes produce identical images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Scale ambiguity** means monocular SLAM can only recover the scene **up to an
unknown scale factor**. To fix the scale, you need external information:
- A known distance between two landmarks
- An IMU (accelerometers measure in meters)
- Known object dimensions

## 35.3 Two View Initialization

Monocular SLAM **cannot start** from a single frame. It needs at least **two views**
with sufficient baseline to initialize the map. The procedure is:

1. Detect features in both frames
2. Match features between frames
3. Estimate the **Essential matrix** $E$ from correspondences
4. Decompose $E = [t]_\times R$ to recover relative pose
5. **Triangulate** 3D points

The result is correct geometry, but at an unknown scale.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(42)
n_pts = 20                       # number of 3D points
baseline = 1.0                   # true baseline (meters)
depth_range = (4.0, 10.0)        # depth range for 3D points
pixel_noise = 0.5                # noise in pixel observations
f = 400.0                        # focal length
# ─────────────────────────────────────────────────────────────────────────────

K = np.array([[f, 0, 320],
              [0, f, 240],
              [0, 0, 1.0]])
K_inv = np.linalg.inv(K)

# Generate 3D points
pts_3d = np.random.uniform(-3, 3, (n_pts, 3))
pts_3d[:, 2] = np.random.uniform(depth_range[0], depth_range[1], n_pts)

# Camera 1: origin. Camera 2: translated along x.
R_true = np.eye(3)
t_true = np.array([baseline, 0, 0])

# Project to both cameras
def project_pts(K, R, t, pts):
    pts_cam = (R @ pts.T).T + t
    pts_norm = pts_cam[:, :2] / pts_cam[:, 2:3]
    px = pts_norm @ K[:2, :2].T + K[:2, 2]
    return px

px1 = project_pts(K, np.eye(3), np.zeros(3), pts_3d) + np.random.normal(0, pixel_noise, (n_pts, 2))
px2 = project_pts(K, R_true, t_true, pts_3d) + np.random.normal(0, pixel_noise, (n_pts, 2))

# Normalize pixel coordinates
def normalize_pixels(K_inv, px):
    px_h = np.column_stack([px, np.ones(len(px))])
    return (K_inv @ px_h.T).T[:, :2]

x1 = normalize_pixels(K_inv, px1)
x2 = normalize_pixels(K_inv, px2)

print(f'Generated {n_pts} 3D points at depths {depth_range}')
print(f'Projected to 2 cameras with baseline = {baseline} m')
print(f'Added {pixel_noise} px noise')

In [ ]:
# Estimate Essential matrix using the 8-point algorithm
def estimate_essential_8pt(x1, x2):
    """Estimate E from normalized correspondences using 8-point algorithm."""
    n = len(x1)
    # Build the constraint matrix A
    # Each correspondence gives: x2^T E x1 = 0
    # With x1 = (u1, v1, 1), x2 = (u2, v2, 1)
    x1h = np.column_stack([x1, np.ones(n)])
    x2h = np.column_stack([x2, np.ones(n)])
    A = np.zeros((n, 9))
    for i in range(n):
        A[i] = np.kron(x2h[i], x1h[i])
    
    # Solve Ae = 0 via SVD
    _, _, Vt = svd(A)
    E_est = Vt[-1].reshape(3, 3)
    
    # Enforce rank-2 constraint
    U, S, Vt2 = svd(E_est)
    S_corrected = np.array([1, 1, 0])  # enforce rank 2
    E_est = U @ np.diag(S_corrected) @ Vt2
    return E_est

E_est = estimate_essential_8pt(x1, x2)

# True essential matrix for comparison
def skew(v):
    return np.array([[0, -v[2], v[1]],
                     [v[2], 0, -v[0]],
                     [-v[1], v[0], 0]])

E_true = skew(t_true) @ R_true
# Normalize both for comparison
E_true_n = E_true / np.linalg.norm(E_true)
E_est_n = E_est / np.linalg.norm(E_est)
# Fix sign ambiguity
if np.sum(E_true_n * E_est_n) < 0:
    E_est_n = -E_est_n

print('Estimated Essential matrix (normalized):')
print(np.round(E_est_n, 4))
print('\nTrue Essential matrix (normalized):')
print(np.round(E_true_n, 4))
print(f'\nFrobenius error: {np.linalg.norm(E_est_n - E_true_n):.6f}')

In [ ]:
# Decompose E to recover R and t
def decompose_essential(E):
    """Decompose E into (R, t). Returns 4 possible solutions."""
    U, S, Vt = svd(E)
    # Ensure proper rotation (det = +1)
    if np.linalg.det(U) < 0:
        U = -U
    if np.linalg.det(Vt) < 0:
        Vt = -Vt
    W = np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1.0]])
    R1 = U @ W @ Vt
    R2 = U @ W.T @ Vt
    t_est = U[:, 2]
    return [(R1, t_est), (R1, -t_est), (R2, t_est), (R2, -t_est)]

solutions = decompose_essential(E_est)

# Triangulate to disambiguate: the correct solution has all points in front of both cameras
def triangulate_point(x1_pt, x2_pt, R, t):
    """Linear triangulation for a single correspondence."""
    # Camera 1: P1 = [I | 0]
    # Camera 2: P2 = [R | t]
    A = np.zeros((4, 4))
    A[0] = x1_pt[0] * np.array([0, 0, 1, 0]) - np.array([1, 0, 0, 0])
    A[1] = x1_pt[1] * np.array([0, 0, 1, 0]) - np.array([0, 1, 0, 0])
    P2 = np.hstack([R, t.reshape(3, 1)])
    A[2] = x2_pt[0] * P2[2] - P2[0]
    A[3] = x2_pt[1] * P2[2] - P2[1]
    _, _, Vt = svd(A)
    X = Vt[-1]
    return X[:3] / X[3]

x1h = np.column_stack([x1, np.ones(n_pts)])
x2h = np.column_stack([x2, np.ones(n_pts)])

best_count = 0
best_idx = 0
for idx, (R_sol, t_sol) in enumerate(solutions):
    count_front = 0
    for j in range(n_pts):
        X = triangulate_point(x1h[j], x2h[j], R_sol, t_sol)
        # Check positive depth in both cameras
        X_cam2 = R_sol @ X + t_sol
        if X[2] > 0 and X_cam2[2] > 0:
            count_front += 1
    if count_front > best_count:
        best_count = count_front
        best_idx = idx

R_rec, t_rec = solutions[best_idx]
print(f'Best solution: {best_idx} ({best_count}/{n_pts} points in front of both cameras)')
print(f'\nRecovered R:\n{np.round(R_rec, 4)}')
print(f'\nRecovered t (unit norm): {np.round(t_rec, 4)}')
print(f'True t (unit norm):      {np.round(t_true / np.linalg.norm(t_true), 4)}')

In [ ]:
# Triangulate all points with the recovered pose
pts_3d_rec = np.zeros((n_pts, 3))
for j in range(n_pts):
    pts_3d_rec[j] = triangulate_point(x1h[j], x2h[j], R_rec, t_rec)

# The reconstruction is correct up to scale. Find the scale.
# Compare pairwise distances
def compute_scale(pts_true, pts_rec):
    """Estimate scale factor s such that pts_rec * s ~ pts_true."""
    scales = []
    for i in range(len(pts_true)):
        for j in range(i+1, len(pts_true)):
            d_true = np.linalg.norm(pts_true[i] - pts_true[j])
            d_rec = np.linalg.norm(pts_rec[i] - pts_rec[j])
            if d_rec > 1e-6:
                scales.append(d_true / d_rec)
    return np.median(scales)

s = compute_scale(pts_3d, pts_3d_rec)
pts_3d_rescaled = pts_3d_rec * s

# Compute RMSE
rmse = np.sqrt(np.mean(np.sum((pts_3d - pts_3d_rescaled)**2, axis=1)))
print(f'Estimated scale factor: {s:.4f}')
print(f'True baseline: {baseline:.2f} m, Recovered baseline: {np.linalg.norm(t_rec):.4f} (unit norm)')
print(f'Reconstruction RMSE after rescaling: {rmse:.4f} m')

Notice that the recovered translation $\mathbf{t}$ has **unit norm**. This is the
manifestation of scale ambiguity: we recover the direction of translation but
not its magnitude. The scale factor must come from external information.

## 35.4 Triangulation Accuracy

Triangulation accuracy depends critically on the **baseline to depth ratio**
$B/Z$. The depth error from triangulation is approximately:

$$\sigma_Z \approx \frac{Z^2}{f \cdot B} \sigma_u$$

where $\sigma_u$ is the pixel noise. Two consequences:
- Depth error grows **quadratically** with depth
- Depth error is **inversely proportional** to baseline

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(42)
focal = 400.0                    # focal length (px)
sigma_px = 1.0                   # pixel noise std
n_trials = 200                   # Monte Carlo trials per setting
baselines = np.linspace(0.05, 3.0, 30)  # baseline values to test
fixed_depth = 10.0               # fixed depth for baseline sweep
depths = np.linspace(2, 50, 30)  # depth values to test
fixed_baseline = 0.5             # fixed baseline for depth sweep
# ─────────────────────────────────────────────────────────────────────────────

def triangulate_depth_mc(f, B, Z, sigma, n_trials):
    """Monte Carlo triangulation depth error."""
    # True point at (0, 0, Z)
    # Camera 1 at origin, Camera 2 at (B, 0, 0)
    # True projections: cam1: (0, 0), cam2: (-fB/Z, 0)
    u1_true = 0.0
    u2_true = -f * B / Z
    
    errors = []
    for _ in range(n_trials):
        u1 = u1_true + np.random.normal(0, sigma)
        u2 = u2_true + np.random.normal(0, sigma)
        disparity = u1 - u2
        if abs(disparity) > 0.01:
            Z_est = f * B / disparity
            errors.append(abs(Z_est - Z))
    return np.mean(errors) if errors else float('inf')

# Sweep baseline at fixed depth
err_vs_baseline = [triangulate_depth_mc(focal, b, fixed_depth, sigma_px, n_trials)
                   for b in baselines]

# Sweep depth at fixed baseline
err_vs_depth = [triangulate_depth_mc(focal, fixed_baseline, d, sigma_px, n_trials)
                for d in depths]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(baselines, err_vs_baseline, 'steelblue', linewidth=2)
# Theoretical: sigma_Z = Z^2 / (f * B) * sigma_px
theory_bl = fixed_depth**2 / (focal * baselines) * sigma_px
ax.plot(baselines, theory_bl, 'tomato', linewidth=2, linestyle='--', label='Theory: $Z^2 \\sigma_u / (fB)$')
ax.set_xlabel('Baseline B (m)', fontsize=12)
ax.set_ylabel('Mean depth error (m)', fontsize=12)
ax.set_title(f'Depth error vs Baseline (Z={fixed_depth}m)', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(0, min(20, max(err_vs_baseline) * 1.1))

ax = axes[1]
ax.plot(depths, err_vs_depth, 'forestgreen', linewidth=2)
theory_d = depths**2 / (focal * fixed_baseline) * sigma_px
ax.plot(depths, theory_d, 'orange', linewidth=2, linestyle='--', label='Theory: $Z^2 \\sigma_u / (fB)$')
ax.set_xlabel('Depth Z (m)', fontsize=12)
ax.set_ylabel('Mean depth error (m)', fontsize=12)
ax.set_title(f'Depth error vs Depth (B={fixed_baseline}m)', fontsize=13)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f'At Z={fixed_depth}m, B=0.1m: error ~ {fixed_depth**2/(focal*0.1)*sigma_px:.2f} m')
print(f'At Z={fixed_depth}m, B=2.0m: error ~ {fixed_depth**2/(focal*2.0)*sigma_px:.2f} m')

In [ ]:
# Heatmap: depth error as function of both baseline and depth
B_grid = np.linspace(0.1, 2.0, 25)
Z_grid = np.linspace(2, 40, 25)
BB, ZZ = np.meshgrid(B_grid, Z_grid)
err_grid = ZZ**2 / (focal * BB) * sigma_px

fig, ax = plt.subplots(figsize=(10, 6))
c = ax.pcolormesh(BB, ZZ, np.log10(err_grid), cmap='RdYlGn_r', shading='auto')
cbar = plt.colorbar(c, ax=ax)
cbar.set_label('log10(depth error in m)', fontsize=11)
# Add contour lines
cs = ax.contour(BB, ZZ, err_grid, levels=[0.1, 0.5, 1.0, 5.0, 10.0],
                colors='black', linewidths=1)
ax.clabel(cs, fmt='%.1f m')
ax.set_xlabel('Baseline B (m)', fontsize=12)
ax.set_ylabel('Depth Z (m)', fontsize=12)
ax.set_title('Theoretical depth error (meters) vs Baseline and Depth', fontsize=13)
plt.tight_layout()
plt.show()

**Practical takeaway:** For monocular SLAM, triangulated points at distances beyond
~20x the baseline are essentially useless. A handheld camera with a few centimeters
of motion between keyframes can only triangulate points within a few meters reliably.

## 35.5 Failure Modes

Monocular SLAM has three classic failure modes:

1. **Pure rotation** produces zero baseline, making triangulation impossible
2. **Forward motion** places the epipole at the image center, making the essential matrix poorly conditioned
3. **Scale drift** causes errors to accumulate over time as scale is re-estimated each frame

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(42)
n_fm_pts = 15                    # points for failure mode demos
rot_angle = 0.1                  # radians, pure rotation
# ─────────────────────────────────────────────────────────────────────────────

# Failure mode 1: Pure rotation
pts_3d_fm = np.random.uniform(-2, 2, (n_fm_pts, 3))
pts_3d_fm[:, 2] = np.random.uniform(4, 10, n_fm_pts)

K = np.array([[400, 0, 320], [0, 400, 240], [0, 0, 1.0]])

# Pure rotation: no translation
R_rot = np.array([[np.cos(rot_angle), 0, np.sin(rot_angle)],
                  [0, 1, 0],
                  [-np.sin(rot_angle), 0, np.cos(rot_angle)]])
t_zero = np.array([0, 0, 0.0])

px1_rot = project_pts(K, np.eye(3), np.zeros(3), pts_3d_fm)
px2_rot = project_pts(K, R_rot, t_zero, pts_3d_fm)

# Try to triangulate
x1_rot = normalize_pixels(K_inv, px1_rot)
x2_rot = normalize_pixels(K_inv, px2_rot)
x1h_rot = np.column_stack([x1_rot, np.ones(n_fm_pts)])
x2h_rot = np.column_stack([x2_rot, np.ones(n_fm_pts)])

tri_depths = []
for j in range(n_fm_pts):
    X = triangulate_point(x1h_rot[j], x2h_rot[j], R_rot, t_zero + 1e-10)
    tri_depths.append(X[2])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ax = axes[0]
ax.scatter(pts_3d_fm[:, 2], tri_depths, c='tomato', s=40)
ax.set_xlabel('True depth (m)', fontsize=11)
ax.set_ylabel('Triangulated depth (m)', fontsize=11)
ax.set_title('Pure Rotation: triangulation fails', fontsize=12, fontweight='bold')
ax.set_ylim(-100, 100)
ax.axhline(0, color='gray', linewidth=0.5)

In [ ]:
# Failure mode 2: Forward motion (epipole at image center)
t_forward = np.array([0, 0, 0.5])  # moving forward along z

px1_fwd = project_pts(K, np.eye(3), np.zeros(3), pts_3d_fm)
px2_fwd = project_pts(K, np.eye(3), t_forward, pts_3d_fm)

# Compute epipolar lines: all pass through the epipole (cx, cy)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(px1_fwd[:, 0], px1_fwd[:, 1], c='steelblue', s=50, zorder=5, label='Frame 1')
ax.scatter(px2_fwd[:, 0], px2_fwd[:, 1], c='tomato', s=50, zorder=5, label='Frame 2')
# Draw flow vectors
for j in range(n_fm_pts):
    ax.annotate('', xy=px2_fwd[j], xytext=px1_fwd[j],
                arrowprops=dict(arrowstyle='->', color='gray', lw=1))
ax.scatter([320], [240], c='orange', s=150, marker='*', zorder=10, label='Epipole (FOE)')
ax.set_xlim(0, 640); ax.set_ylim(480, 0)
ax.set_title('Forward motion: all flow radiates from center', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Compare with lateral motion
t_lateral = np.array([0.5, 0, 0])
px2_lat = project_pts(K, np.eye(3), t_lateral, pts_3d_fm)

ax = axes[1]
ax.scatter(px1_fwd[:, 0], px1_fwd[:, 1], c='steelblue', s=50, zorder=5, label='Frame 1')
ax.scatter(px2_lat[:, 0], px2_lat[:, 1], c='forestgreen', s=50, zorder=5, label='Frame 2')
for j in range(n_fm_pts):
    ax.annotate('', xy=px2_lat[j], xytext=px1_fwd[j],
                arrowprops=dict(arrowstyle='->', color='gray', lw=1))
ax.set_xlim(0, 640); ax.set_ylim(480, 0)
ax.set_title('Lateral motion: diverse flow directions (good)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(42)
n_frames = 40                    # number of frames for scale drift demo
scale_noise_std = 0.03           # noise in per-frame scale estimation
# ─────────────────────────────────────────────────────────────────────────────

# Failure mode 3: Scale drift
# Simulate monocular VO: each frame estimates motion up to a noisy scale
true_speed = 1.0  # constant speed
true_positions = np.cumsum(np.ones(n_frames) * true_speed)

# Monocular: scale estimated with noise each frame, errors compound
estimated_scales = 1.0 + np.random.normal(0, scale_noise_std, n_frames)
estimated_positions = np.cumsum(estimated_scales * true_speed)

# Scale error accumulates
scale_error = (estimated_positions - true_positions) / true_positions * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(range(n_frames), true_positions, 'forestgreen', linewidth=2, label='True position')
ax.plot(range(n_frames), estimated_positions, 'tomato', linewidth=2, label='Monocular estimate')
ax.fill_between(range(n_frames), true_positions, estimated_positions, alpha=0.2, color='tomato')
ax.set_xlabel('Frame', fontsize=12)
ax.set_ylabel('Position (m)', fontsize=12)
ax.set_title('Scale drift: monocular position diverges', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

ax = axes[1]
ax.plot(range(n_frames), scale_error, 'steelblue', linewidth=2)
ax.axhline(0, color='gray', linewidth=0.5)
ax.set_xlabel('Frame', fontsize=12)
ax.set_ylabel('Position error (%)', fontsize=12)
ax.set_title('Cumulative scale error grows with time', fontsize=13)

plt.tight_layout()
plt.show()

print(f'After {n_frames} frames:')
print(f'  True distance traveled: {true_positions[-1]:.1f} m')
print(f'  Estimated distance:     {estimated_positions[-1]:.1f} m')
print(f'  Scale error:            {scale_error[-1]:.1f}%')

**Summary of failure modes:**

| Mode | Cause | Symptom |
|------|-------|--------|
| Pure rotation | Zero baseline | Triangulation produces garbage depths |
| Forward motion | Epipole at image center | Essential matrix poorly conditioned |
| Scale drift | Per frame scale noise | Position error grows without bound |

## Capstone: Full Monocular Initialization

We now put everything together. Given two views of a room with 30 3D points:
1. Estimate the Essential matrix
2. Recover the relative pose
3. Triangulate all points
4. Rescale using one known distance
5. Compare with ground truth

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(123)
n_cap_pts = 30                   # number of 3D points
room_size = (6, 4, 8)            # room dimensions (x, y, z)
true_baseline_cap = 0.8          # baseline between cameras
true_angle_cap = 0.05            # slight rotation (radians)
noise_cap = 0.5                  # pixel noise
f_cap = 500.0                    # focal length
# ─────────────────────────────────────────────────────────────────────────────

K_cap = np.array([[f_cap, 0, 320], [0, f_cap, 240], [0, 0, 1.0]])
K_cap_inv = np.linalg.inv(K_cap)

# Generate room points
pts_room = np.column_stack([
    np.random.uniform(-room_size[0]/2, room_size[0]/2, n_cap_pts),
    np.random.uniform(-room_size[1]/2, room_size[1]/2, n_cap_pts),
    np.random.uniform(3, room_size[2], n_cap_pts)
])

# Camera 2 pose: slight rotation + translation
R_cap = np.array([[np.cos(true_angle_cap), 0, np.sin(true_angle_cap)],
                  [0, 1, 0],
                  [-np.sin(true_angle_cap), 0, np.cos(true_angle_cap)]])
t_cap = np.array([true_baseline_cap, 0.0, 0.1])

# Project
px1_cap = project_pts(K_cap, np.eye(3), np.zeros(3), pts_room)
px2_cap = project_pts(K_cap, R_cap, t_cap, pts_room)

# Add noise
px1_cap_noisy = px1_cap + np.random.normal(0, noise_cap, px1_cap.shape)
px2_cap_noisy = px2_cap + np.random.normal(0, noise_cap, px2_cap.shape)

# Step 1: Estimate E
x1_cap = normalize_pixels(K_cap_inv, px1_cap_noisy)
x2_cap = normalize_pixels(K_cap_inv, px2_cap_noisy)
E_cap = estimate_essential_8pt(x1_cap, x2_cap)

# Step 2: Decompose and select correct solution
solutions_cap = decompose_essential(E_cap)
x1h_cap = np.column_stack([x1_cap, np.ones(n_cap_pts)])
x2h_cap = np.column_stack([x2_cap, np.ones(n_cap_pts)])

best_count_cap = 0
best_idx_cap = 0
for idx, (R_s, t_s) in enumerate(solutions_cap):
    cnt = 0
    for j in range(n_cap_pts):
        X = triangulate_point(x1h_cap[j], x2h_cap[j], R_s, t_s)
        X2 = R_s @ X + t_s
        if X[2] > 0 and X2[2] > 0:
            cnt += 1
    if cnt > best_count_cap:
        best_count_cap = cnt
        best_idx_cap = idx

R_cap_rec, t_cap_rec = solutions_cap[best_idx_cap]

# Step 3: Triangulate
pts_cap_rec = np.zeros((n_cap_pts, 3))
for j in range(n_cap_pts):
    pts_cap_rec[j] = triangulate_point(x1h_cap[j], x2h_cap[j], R_cap_rec, t_cap_rec)

print(f'Pose recovery: {best_count_cap}/{n_cap_pts} points in front of both cameras')

In [ ]:
# Step 4: Rescale using one known distance (distance between points 0 and 1)
known_dist = np.linalg.norm(pts_room[0] - pts_room[1])
rec_dist = np.linalg.norm(pts_cap_rec[0] - pts_cap_rec[1])
scale_cap = known_dist / rec_dist

pts_cap_scaled = pts_cap_rec * scale_cap

# Step 5: Compare with ground truth
errors = np.linalg.norm(pts_room - pts_cap_scaled, axis=1)
rmse_cap = np.sqrt(np.mean(errors**2))

fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(131, projection='3d')
ax1.scatter(pts_room[:, 0], pts_room[:, 1], pts_room[:, 2], c='steelblue', s=40)
ax1.set_title('Ground truth', fontsize=12)
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

ax2 = fig.add_subplot(132, projection='3d')
ax2.scatter(pts_cap_scaled[:, 0], pts_cap_scaled[:, 1], pts_cap_scaled[:, 2],
            c='forestgreen', s=40)
ax2.set_title('Rescaled reconstruction', fontsize=12)
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')

ax3 = fig.add_subplot(133)
ax3.bar(range(n_cap_pts), errors, color='tomato', alpha=0.7)
ax3.axhline(rmse_cap, color='steelblue', linewidth=2, linestyle='--', label=f'RMSE = {rmse_cap:.3f} m')
ax3.set_xlabel('Point index', fontsize=11)
ax3.set_ylabel('3D error (m)', fontsize=11)
ax3.set_title('Per point reconstruction error', fontsize=12)
ax3.legend(fontsize=11)

plt.suptitle('Monocular Initialization: Two View Reconstruction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n=== CAPSTONE RESULTS ===')
print(f'Known distance used for rescaling: {known_dist:.3f} m')
print(f'Estimated scale factor: {scale_cap:.4f}')
print(f'Scale corrected RMSE: {rmse_cap:.4f} m')
print(f'Mean error: {np.mean(errors):.4f} m')
print(f'Max error:  {np.max(errors):.4f} m')

**Capstone takeaways:**
- The 8 point algorithm recovers $E$ reliably with enough points and low noise
- Pose decomposition gives 4 solutions; only one places all points in front of both cameras
- Triangulation accuracy depends on the baseline to depth ratio
- One known distance is sufficient to resolve the scale ambiguity
- The rescaled reconstruction matches ground truth well, with errors concentrated on distant points

---

## Exercises

### Exercise 35.1
Modify the scale ambiguity demo (Section 35.2) to use **three** different scale factors.
Verify that all three scenes produce identical pixel coordinates in both cameras.
Plot all three scenes in 3D with different colors.

In [ ]:
# Your code here

### Exercise 35.2
In the triangulation section, add **outlier noise** to 20% of pixel observations (shift
by 50px). Run the 8 point algorithm and observe how the reconstruction degrades.
Then implement RANSAC around the 8 point algorithm and show improved results.

In [ ]:
# Your code here

### Exercise 35.3
Create a simulation where the camera moves in a circle (10 frames), triangulating
points between consecutive frames. Plot the per frame scale estimate and show how
scale drift accumulates. Compare with the true trajectory.

In [ ]:
# Your code here

### Exercise 35.4
Implement a function that, given pixel noise $\sigma_u$, focal length $f$,
and desired depth accuracy $\sigma_Z$, computes the **minimum baseline** needed
at a given depth. Plot minimum baseline vs depth for $\sigma_Z = 0.1, 0.5, 1.0$ m.

In [ ]:
# Your code here

### Exercise 35.5
The capstone uses distance between two points to fix scale. Implement an alternative
that uses the **known height of the camera** above the ground plane. Assume the ground
plane is $Y = 0$ and the camera is at height $h = 1.5$ m. Show that this also resolves
the scale ambiguity.

In [ ]:
# Your code here